<a href="https://colab.research.google.com/github/busycaesar/Finetune_LoRA/blob/Master/gemma_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.
os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

In [2]:
%pip install -U -q keras-hub keras

In [3]:
import keras
import keras_hub
import numpy as np

In [4]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("gemma3_270m")

In [ ]:
gemma_lm.generate("I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.", max_length=500)

'I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swollen foot and I have a very swollen ankle. I have a very swoll

In [5]:
from datasets import load_dataset

ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")

# Convert to pandas and sample
df = ds.to_pandas().sample(100, random_state=42)

features = {
    "prompts": df["input"].tolist(),
    "responses": df["output"].tolist()
}

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
print(len(features['prompts']))
print(features['prompts'][0])
print(features['responses'][0])

100
I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.
Dear patient Here are the possibilities of what you might have.1)PhlebitisPhlebitis means inflammation of the veins, and can cause redness, itching, irritation, pain, and swelling. A simple Doppler can rule this out.2Blood clot in the lifeblood clots in the leg can become very dangerous, symptoms include swelling, redness, tenderness in the leg. Coagulation profile with an angiography may be required3)Cellulitis


In [7]:
gemma_lm.backbone.enable_lora(rank=4)

In [ ]:
#gemma_lm.preprocessor.sequence_length = 512

In [ ]:
gemma_lm.fit(features, epochs=1, batch_size=1)

  2/100 ━━━━━━━━━━━━━━━━━━━━ 1:06:05 40s/step - loss: 0.3185 - sparse_categorical_accuracy: 0.3639

In [ ]:
gemma_lm.generate("I wake in the night, usually about 2-3 hours after going to sleep, with both feet and legs to mid calf feeling like they are on fire. slight red discolorization, minor swelling. This is very painful but after getting up, I can walk it off in about 30 minutes.", max_length=500)

In [ ]:
gemma_lm.load_lora_weights("lora_weights.h5")